In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F

You can train your data on books or data sets from wikipedia pages or https://www.gutenberg.org/

In [2]:
text = '''
Artificial intelligence is the future of technology.
Machines can learn from data and improve over time.
Humans and machines can work together to solve problems.
'''

In [3]:
words = list(set(text.lower().split()))
vocab = {w: i for i, w in enumerate(words)}
reverse_vocab = {i: w for w, i in vocab.items()}
vocab_size = len(vocab)

In [5]:
def encode(sentence):
    return [vocab[w] for w in sentence.lower().split()]

tokens = encode(text)
print(tokens)

[1, 9, 10, 15, 11, 7, 6, 4, 3, 5, 2, 21, 19, 0, 17, 12, 18, 19, 4, 3, 8, 13, 14, 20, 16]


In [6]:
seq_length = 4

X, y = [],[]
for i in range(len(tokens) - seq_length):
    X.append(tokens[i:i+seq_length])
    y.append(tokens[i:i+seq_length])

X = torch.tensor(X)
y = torch.tensor(y)

In [7]:
class BabyGPT(nn.Module):
    def __init__(self, vocab_size, d_model=64):
        super().__init__()

        self.embedding = nn.Embedding(vocab_size, d_model)

        self.q = nn.Linear(d_model, d_model)
        self.k = nn.Linear(d_model, d_model)
        self.v = nn.Linear(d_model, d_model)

        self.fc_out = nn.Linear(d_model, vocab_size)

    def forward(self, x):
        x = self.embedding(x)  # (batch, seq, d_model)

        Q = self.q(x)
        K = self.k(x)
        V = self.v(x)

        scores = torch.matmul(Q, K.transpose(-2, -1))
        scores = scores / (x.size(-1) ** 0.5)

        weights = F.softmax(scores, dim=-1)
        attention = torch.matmul(weights, V)

        out = attention[:, -1, :]

        return self.fc_out(out)

In [9]:
model = BabyGPT(vocab_size)
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
loss_fn = nn.CrossEntropyLoss()

# Create the y_targets tensor for CrossEntropyLoss
y_targets = []
for i in range(len(tokens) - seq_length):
    y_targets.append(tokens[i + seq_length])
y_targets = torch.tensor(y_targets)

for epoch in range(300):
    optimizer.zero_grad()

    output = model(X)
    loss = loss_fn(output, y_targets)

    loss.backward()
    optimizer.step()

    if epoch % 50 == 0:
        print(f"Epoch {epoch}, Loss: {loss.item():.4f}")

Epoch 0, Loss: 3.0637
Epoch 50, Loss: 0.5639
Epoch 100, Loss: 0.1480
Epoch 150, Loss: 0.0856
Epoch 200, Loss: 0.0758
Epoch 250, Loss: 0.0721


In [10]:
def generate(start_words, steps=10):
    model.eval()

    words = start_words.copy()

    for _ in range(steps):
        x = torch.tensor([words[-seq_length:]])
        out = model(x)

        next_word = torch.argmax(out).item()
        words.append(next_word)

    return " ".join([reverse_vocab[i] for i in words])

In [11]:
start = tokens[:seq_length]
print("\nGenerated Text:\n")
print(generate(start))


Generated Text:

artificial intelligence is the future of technology. machines can work together to solve problems.
